# Defense eVTOL Trajectory Optimization Dataset — Tutorial

This notebook provides a guided tour of the four-layer dataset:
1. **Perception** — 1,057,714 grid cells with terrain, wind, obstacle, and SAM threat data
2. **Planning** — 2,000 RRT* + NSGA-III Pareto-optimal trajectories
3. **Vehicle** — 2,000 physics simulation records (6-DoF, BEMT rotor, battery model)
4. **Control** — 2,000 closed-loop PID simulation records (50 Hz cascade)

By the end you will:
- Load all four datasets and understand their schemas
- Visualise the perception cost field and threat landscape
- Analyse trajectory-level statistics
- Train and evaluate three baseline ML models
- Reproduce the NeurIPS baseline results table

**Reference:** Ramakrishnan, A. (2026). *Trajectory Optimization in Defense eVTOLs.*

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Dataset paths — adjust if running from a different directory
REPO_ROOT = Path('..').resolve()
OUTPUTS   = REPO_ROOT / 'outputs'

PERC_PATH  = OUTPUTS / 'perception_dataset' / 'perception_full_dataset.csv'
PLAN_PATH  = OUTPUTS / 'planning_dataset'   / 'test_final.parquet'
VEH_PATH   = OUTPUTS / 'vehicle'            / 'vehicle_dataset.parquet'
CTRL_PATH  = OUTPUTS / 'control'            / 'control_dataset.parquet'

# Check files exist
for p in [PERC_PATH, PLAN_PATH, VEH_PATH, CTRL_PATH]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {p.name}')

## 1. Perception Layer

The perception dataset covers a **33 × 55 km operating area** (Delhi NCR, 28.40–28.90°N, 76.90–77.50°E) 
across **14 altitude bands** (50–1500 m MSL), producing **1,057,714 grid cells** at 222 m spatial resolution.

In [ ]:
# Load a small subset for exploration (full file is ~180 MB)
df_perc = pd.read_csv(PERC_PATH, nrows=100_000)
print(f'Loaded {len(df_perc):,} rows, {df_perc.shape[1]} columns')
df_perc.describe().T[['mean','std','min','max']].round(4)

In [ ]:
# Visualise the fused cost field at 200 m altitude
df_200 = pd.read_csv(PERC_PATH).query('alt_m == 200')
print(f'{len(df_200):,} cells at 200 m altitude')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, cmap, label in zip(
    axes,
    ['terrain_cost', 'threat_cost', 'fused_cost'],
    ['terrain', 'YlOrRd', 'viridis'],
    ['Terrain Cost', 'Threat Cost', 'Fused Cost (mission planning signal)'],
):
    sc = ax.scatter(df_200['lon'], df_200['lat'],
                    c=df_200[col], cmap=cmap, s=1, vmin=0, vmax=1)
    plt.colorbar(sc, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')

plt.suptitle('Perception Cost Fields at 200 m MSL', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Check threat probability distribution across altitude bands
df_all = pd.read_csv(PERC_PATH, usecols=['alt_m', 'combined_threat_prob', 'T1_detect_prob',
                                          'T2_detect_prob', 'T3_detect_prob'])
threat_by_alt = df_all.groupby('alt_m')[['combined_threat_prob']].agg(['mean','std'])
threat_by_alt.columns = ['mean','std']
print('Threat probability by altitude (note saturation at 1.0):')
print(threat_by_alt.to_string())

> **Note:** `combined_threat_prob` saturates at 1.0 everywhere because the three SAM emitters cover the entire operating area. 
> The planning layer uses a **distance-decay threat gradient** computed from emitter positions that produces meaningful spatial variation (std ≈ 0.12). 
> See `doc/research_limitations.md §1.1` and `scripts/planning/dataset.py::_compute_threat_gradient()`.

## 2. Planning Layer

Each record represents one RRT*-planned trajectory evaluated by NSGA-III across three objectives:
- **Energy** (Wh) — estimated from momentum theory + climb work
- **Time** (s) — path length / cruise speed
- **Threat** — integrated distance-decay detection probability

In [ ]:
df_plan = pd.read_parquet(PLAN_PATH)
print(f'Planning dataset: {df_plan.shape}')
print(f'Risk label distribution: {df_plan["risk_label"].value_counts().to_dict()}')
df_plan[['path_length_m','time_cost_s','energy_cost_wh','threat_cost',
         'fused_cost_mean','risk_label']].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Path length distribution
axes[0].hist(df_plan['path_length_m'] / 1000, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Path Length (km)')
axes[0].set_ylabel('Count')
axes[0].set_title('Path Length Distribution')

# Threat cost distribution
axes[1].hist(df_plan['threat_cost'], bins=40, color='tomato', edgecolor='white')
axes[1].set_xlabel('Threat Cost (mean P_detect)')
axes[1].set_title('Threat Cost Distribution (distance-decay gradient)')

# Energy vs time scatter coloured by risk
colors = ['green' if r == 0 else 'red' for r in df_plan['risk_label']]
axes[2].scatter(df_plan['time_cost_s'], df_plan['energy_cost_wh'],
                c=colors, alpha=0.4, s=10)
axes[2].set_xlabel('Time Cost (s)')
axes[2].set_ylabel('Energy Cost (Wh)')
axes[2].set_title('Energy vs Time (red=high-risk, green=low-risk)')

plt.tight_layout()
plt.show()

## 3. Vehicle Layer

The vehicle dataset simulates each planning trajectory through a high-fidelity 6-DoF physics model 
(BEMT rotor, detailed aerodynamics, Coulomb-counting battery) without closing the control loop.

In [ ]:
df_veh = pd.read_parquet(VEH_PATH)
print(f'Vehicle dataset: {df_veh.shape}')

# Key physics metrics
print('\nSOC statistics:')
print(df_veh[['soc_initial','soc_final','soc_minimum','energy_consumed_wh']].describe().round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Acoustic signature
if 'spl_hover_a_dB' in df_veh.columns:
    axes[0,0].hist(df_veh['spl_hover_a_dB'], bins=30, color='purple', edgecolor='white')
    axes[0,0].set_xlabel('SPL Hover A-weighted (dB(A))')
    axes[0,0].set_title('Acoustic Signature Distribution')

# RCS
if 'rcs_cruise_x_dBsm' in df_veh.columns:
    axes[0,1].hist(df_veh['rcs_cruise_x_dBsm'], bins=30, color='navy', edgecolor='white')
    axes[0,1].set_xlabel('RCS Cruise X-band (dBsm)')
    axes[0,1].set_title('Radar Cross Section Distribution')

# Energy consumed
axes[0,2].hist(df_veh['energy_consumed_wh'], bins=30, color='darkorange', edgecolor='white')
axes[0,2].set_xlabel('Energy Consumed (Wh)')
axes[0,2].set_title('Energy Consumption Distribution')

# SOC trajectory
axes[1,0].scatter(df_veh['soc_initial'], df_veh['soc_final'], alpha=0.3, s=5)
axes[1,0].plot([0.5,1.0],[0.5,1.0], 'r--', label='SOC unchanged')
axes[1,0].set_xlabel('Initial SOC')
axes[1,0].set_ylabel('Final SOC')
axes[1,0].set_title('Battery State of Charge')
axes[1,0].legend()

# Rotor efficiency
if 'hover_efficiency_mean' in df_veh.columns:
    axes[1,1].hist(df_veh['hover_efficiency_mean'], bins=30, color='green', edgecolor='white')
    axes[1,1].set_xlabel('Hover Efficiency')
    axes[1,1].set_title('Mean Hover Efficiency')
else:
    axes[1,1].text(0.5, 0.5, 'hover_efficiency_mean\nnot in dataset', ha='center', va='center',
                   transform=axes[1,1].transAxes, fontsize=12)

# Cruise speed
if 'cruise_speed_ms' in df_veh.columns:
    axes[1,2].hist(df_veh['cruise_speed_ms'], bins=30, color='teal', edgecolor='white')
    axes[1,2].set_xlabel('Cruise Speed (m/s)')
    axes[1,2].set_title('Cruise Speed Distribution')
else:
    axes[1,2].text(0.5, 0.5, 'cruise_speed_ms\nnot in dataset', ha='center', va='center',
                   transform=axes[1,2].transAxes, fontsize=12)

plt.suptitle('Vehicle Layer — Key Physics Metrics', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Control Layer

The control dataset runs a 5-loop cascaded PID autopilot at 50 Hz for each mission,
recording tracking error metrics, actuator statistics, and closed-loop energy.

In [ ]:
df_ctrl = pd.read_parquet(CTRL_PATH)
print(f'Control dataset: {df_ctrl.shape}')

# Tracking performance summary
tracking_cols = ['alt_error_mean_m','vel_error_mean_ms','att_error_mean_rad',
                 'pos_error_mean_m','itae_alt','itae_pos']
available = [c for c in tracking_cols if c in df_ctrl.columns]
print('\nTracking error statistics:')
print(df_ctrl[available].describe().round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Altitude error
if 'alt_error_mean_m' in df_ctrl.columns:
    axes[0,0].hist(df_ctrl['alt_error_mean_m'], bins=40, color='steelblue', edgecolor='white')
    axes[0,0].set_xlabel('Altitude Error (m)')
    axes[0,0].set_title('Mean Altitude Tracking Error')
    axes[0,0].axvline(df_ctrl['alt_error_mean_m'].mean(), color='red', linestyle='--',
                      label=f"mean={df_ctrl['alt_error_mean_m'].mean():.2f} m")
    axes[0,0].legend()

# Velocity error
if 'vel_error_mean_ms' in df_ctrl.columns:
    axes[0,1].hist(df_ctrl['vel_error_mean_ms'], bins=40, color='darkorange', edgecolor='white')
    axes[0,1].set_xlabel('Velocity Error (m/s)')
    axes[0,1].set_title('Mean Velocity Tracking Error')

# Thrust command
if 'thrust_cmd_mean_N' in df_ctrl.columns:
    axes[0,2].hist(df_ctrl['thrust_cmd_mean_N'], bins=40, color='purple', edgecolor='white')
    axes[0,2].set_xlabel('Mean Thrust Command (N)')
    axes[0,2].set_title('Thrust Command Distribution')

# Motor balance
if 'motor_T_balance_N' in df_ctrl.columns:
    axes[1,0].hist(df_ctrl['motor_T_balance_N'], bins=40, color='green', edgecolor='white')
    axes[1,0].set_xlabel('Motor Thrust Imbalance (N)')
    axes[1,0].set_title('Motor Thrust Balance')

# Energy consumed
if 'energy_consumed_wh' in df_ctrl.columns:
    axes[1,1].hist(df_ctrl['energy_consumed_wh'], bins=40, color='tomato', edgecolor='white')
    axes[1,1].set_xlabel('Energy Consumed (Wh)')
    axes[1,1].set_title('Closed-Loop Energy Consumption')

# PWM utilisation
if 'pwm_utilisation_pct' in df_ctrl.columns:
    axes[1,2].hist(df_ctrl['pwm_utilisation_pct'], bins=40, color='teal', edgecolor='white')
    axes[1,2].set_xlabel('PWM Utilisation (%)')
    axes[1,2].set_title('Actuator Utilisation')

plt.suptitle('Control Layer — Tracking Error & Actuator Metrics', fontsize=13)
plt.tight_layout()
plt.show()

## 5. ML Task 1 — Risk Label Classification

**Goal:** Predict whether a trajectory has high environmental risk (`risk_label = 1`) from path geometry and cost field features.

**Use case:** Fast online risk screening before running the full physics simulation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, classification_report

PLAN_FEATURES = [
    'path_length_m', 'n_waypoints', 'time_cost_s', 'energy_cost_wh',
    'threat_cost', 'terrain_cost_mean', 'wind_cost_mean',
    'obstacle_cost_mean', 'fused_cost_mean', 'max_combined_threat',
]

feats = [f for f in PLAN_FEATURES if f in df_plan.columns]
X = df_plan[feats].fillna(df_plan[feats].median()).to_numpy()
y = df_plan['risk_label'].to_numpy()

print(f'Features: {feats}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                           learning_rate=0.05, random_state=42)),
    ]),
    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)),
    ]),
}

results_t1 = {}
for name, pipe in models.items():
    y_prob = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y, y_prob)
    f1  = f1_score(y, y_pred)
    results_t1[name] = {'AUC-ROC': auc, 'F1': f1}
    print(f'{name:<25}  AUC={auc:.4f}  F1={f1:.4f}')

## 6. ML Task 2 — Energy Consumption Regression

**Goal:** Predict closed-loop energy consumption from mission parameters and control metrics.

**Use case:** Energy-aware mission planning without running the full simulation.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error

ENERGY_FEATURES = [
    'path_length_m', 'mission_time_s', 'cruise_speed_ref_ms', 'cruise_altitude_ref_m',
    'n_waypoints', 'hover_frac_ctrl', 'cruise_frac_ctrl',
    'thrust_cmd_mean_N', 'thrust_cmd_std_N', 'motor_T_mean_N',
    'pwm_mean_us', 'pwm_utilisation_pct', 'soc_initial',
]

feats_e = [f for f in ENERGY_FEATURES if f in df_ctrl.columns]
X_e = df_ctrl[feats_e].fillna(df_ctrl[feats_e].median()).to_numpy()
y_e = df_ctrl['energy_consumed_wh'].to_numpy()

cv_r = KFold(n_splits=5, shuffle=True, random_state=42)

reg_models = {
    'Ridge': Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=1.0))]),
    'Gradient Boosting': Pipeline([('scaler', StandardScaler()),
        ('reg', GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                          learning_rate=0.05, random_state=42))]),
    'MLP': Pipeline([('scaler', StandardScaler()),
        ('reg', MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))]),
}

results_t2 = {}
for name, pipe in reg_models.items():
    y_pred = cross_val_predict(pipe, X_e, y_e, cv=cv_r)
    r2  = r2_score(y_e, y_pred)
    mae = mean_absolute_error(y_e, y_pred)
    results_t2[name] = {'R2': r2, 'MAE_Wh': mae}
    print(f'{name:<25}  R²={r2:.4f}  MAE={mae:.2f} Wh')

## 7. Results Summary

Reproduce the NeurIPS baseline results table.

In [ ]:
print('='*70)
print('BASELINE RESULTS — 5-Fold Cross-Validation')
print('='*70)
print(f'{'Task':<32} {'Model':<25} {'Metric1':>10} {'Metric2':>10}')
print('-'*70)

for name, res in results_t1.items():
    print(f'{"T1:RiskClassification":<32} {name:<25} {"AUC-ROC"[0:3]}={res["AUC-ROC"]:.4f} F1={res["F1"]:.4f}')

for name, res in results_t2.items():
    print(f'{"T2:EnergyRegression":<32} {name:<25} R²={res["R2"]:.4f}  MAE={res["MAE_Wh"]:.1f}Wh')

print('='*70)

## 8. Feature Importance Analysis

Which perception features best predict trajectory risk?

In [ ]:
# Train GBM on full dataset to get feature importances
from sklearn.ensemble import GradientBoostingClassifier

feats = [f for f in PLAN_FEATURES if f in df_plan.columns]
X_all = df_plan[feats].fillna(df_plan[feats].median()).to_numpy()
y_all = df_plan['risk_label'].to_numpy()

gbm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)),
])
gbm.fit(X_all, y_all)

importances = gbm.named_steps['clf'].feature_importances_
feat_imp = pd.Series(importances, index=feats).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
feat_imp.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('GBM Feature Importance — Risk Label Classification')
ax.set_ylabel('Importance')
ax.set_xlabel('Feature')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
print(feat_imp.head())

## 9. Train / Val / Test Splits

Load the pre-computed 80/10/10 stratified index splits for reproducibility.

In [ ]:
SPLITS_DIR = OUTPUTS / 'splits'

if SPLITS_DIR.exists():
    for name in ['planning', 'control']:
        for split in ['train', 'val', 'test']:
            path = SPLITS_DIR / f'{name}_{split}_idx.npy'
            if path.exists():
                idx = np.load(path)
                print(f'{name} {split}: {len(idx):,} samples')
            else:
                print(f'{path.name} NOT FOUND — run scripts/ml/baseline_experiments.py')
else:
    print('Splits directory not found. Run scripts/ml/baseline_experiments.py to generate splits.')

## 10. Citation

If you use this dataset, please cite:

```bibtex
@misc{ramakrishnan2026evtol,
  author       = {Ramakrishnan, Aditi},
  title        = {Trajectory Optimization in Defense eVTOLs:
                  An Open Multi-Objective Dataset and Benchmark},
  year         = {2026},
  note         = {NeurIPS Datasets \& Benchmarks submission}
}
```

**Data sources:**
- Terrain: NASA SRTM (public domain)
- Wind: Open-Meteo (CC-BY 4.0)
- Obstacles: OpenStreetMap (ODbL)
- Threat model: Mahafza (2005) radar equation — published textbook methodology